# CycloneDX SBOM Analysis Jupyter Notebook

This jupyter notebook was developed as part of the supporting materials for the 2024 CISA SBOM Plugfest. The purpose of this notebook is to ingest CycloneDX SBOMs, and extract information from them as a batch, and export summary and component information in csv files.

**Packages:**
- *os*: for traversing file directories
- *json*: deserializing json files
- *pandas*: building dataframes
- *networkx*: graph algorithms (depth, paths)
- *collections.defaultdict*: instantitating dictionaries
- *matplotlib.pyplot*: optional plotting
- *networkx.drawing.nx_pydot.graphviz_layout*: optional plotting
- *pathlib*: creating directories and path objects

*shared_functions* is a separate file hosting a number of functions needed for processing SBOMs. Our intention was to migrate many functions in notebooks into separate python files and to convert notebooks into python scripts, but we ran out of time. Note: shared_functions.py should be co-located with this notebook.


## Recommended Directory Structure   
This directory structure was used to facilitate processing of .json sboms of each type (CycloneDX, SPDX). There are other more efficient ways to do this, but this is the methodology we used. As written, these notebooks should be run from the ```code``` location. Additionally, although only two targets are illustrated here, this structure should be replicated for sboms corresponding to additional targets.
```
📂project
┣ 📂code
┃ ┣ 📜cyclonedx_analyzer.ipynb
┃ ┣ 📜shared_functions.py
┃ ┗ 📜spdx_analyzer.ipynb
┣ 📂outputs
┣ 📂submissions_by_target
┃ ┣ 📂target_1
┃ ┃ ┣ 📂build
┃ ┃ ┃ ┣ 📂cyclone
┃ ┃ ┃ ┃ ┗ 📜cdx_build_target_2_sbom_1.json
┃ ┃ ┃ ┗ 📂spdx
┃ ┃ ┃   ┗ 📜spdx_build_target_2_sbom_1.json
┃ ┃ ┗ 📂source
┃ ┃   ┣ 📂cyclone
┃ ┃   ┃ ┗ 📜cdx_source_target_2_sbom_1.json
┃ ┃   ┗ 📂spdx
┃ ┃     ┗ 📜spdx_source_target_2_sbom_1.json
┃ ┗ 📂target_2
┃   ┣ 📂build
┃   ┃ ┣ 📂cyclone
┃   ┃ ┃ ┗ 📜cdx_build_target_2_sbom_1.json
┃   ┃ ┗ 📂spdx
┃   ┃   ┗ 📜spdx_build_target_2_sbom_1.json
┃   ┗ 📂source
┃     ┣ 📂cyclone
┃     ┃ ┗ 📜cdx_source_target_2_sbom_1.json
┃     ┗ 📂spdx
┃       ┗ 📜spdx_source_target_2_sbom_1.json
┗ 📂baseline_sboms
     ┣ 📂target_1
     ┃ ┗ 📜target_1_baseline_sbom_1.json
     ┗ 📂target_2
       ┗ 📜target_2_baseline_sbom_1.json
```



## Outputs

After running all cells, you will produce:
- ```(bld/src)_comp_version_(target).csv```: files that summarize how many CycloneDX SBOMs of type (target; source or build) each (component, version) was found in. There will be one of these files produced for each phase (build/source) for each target.

- ```merged_cycloneDX_component_files.csv```: a file that combines all of the data from component version files (see previous bullet).

- ```(bld/src)_full_combined_(target).csv```: files that summarize information extracted from each component of all build/source sboms for each target. One file is produced for each phase (build/source) for each target. 

- ```(bld/src)_(target)_cyclonedx_summary.csv```: files that summarize top-level information about each sbom (primarily, minimum elements). There will be one of these files produced for each phase (build/source) for each target.

- ```merged_cycloneDX_summary_files.csv```: a file that combines all of the data from individual summary files (see previous bullet).


## Import Statement

In [ ]:
## Import packages ##

import os
import json
import pandas as pd
import networkx as nx
from collections import defaultdict
from matplotlib import pyplot as plt
from networkx.drawing.nx_pydot import graphviz_layout
from pathlib import Path

from shared_functions import *


## Custom Functions for Parsing CycloneDX SBOMs

In [ ]:
def getLicenses(element):
    """ Get a list of licenses from the given json element, which is expected to be a dict """
    licenses = []
    license_elems = element.get("licenses", [])
    for elem in license_elems:
        if "expression" in elem and elem.get("expression", "").strip() != "":
            licenses.append(elem.get("expression", "").strip())
        elif "license" in elem:
            license = elem.get("license", {})
            if "id" in license:
                licenses.append(license.get("id"))
            elif "name" in license:
                licenses.append(license.get("name"))
    return licenses


def getPhases(element):
    """ Get a list of lifecycle phases """
    phases = []
    lifecycle_elems = element.get('lifecycles', [])
    for elem in lifecycle_elems:
        if "phase" in elem:
            phases.append(elem["phase"])
        elif "name" in elem:
            phases.append(elem["name"])
    return phases


def getAuthors(element):
    """ Get a list of author dicts with name, email, phone, if present """
    authors = 'False'
    author_elems = element.get('authors', [])
    for elem in author_elems:
        author = {val: elem[val] for val in ['name', 'email', 'phone'] if val in elem}
        if len(author) > 0:
            # authors.append(author)
            authors = 'True'
    return authors


def getHashes(element):
    """ Get a list of the hash alg and content """
    hashes = []
    hash_elems = element.get("hashes", [])
    for elem in hash_elems:
        if "alg" in elem and "content" in elem:
            hashes.append({"alg": elem["alg"], "content": elem["content"]})
    return hashes


def getSwid(component):
    """ Returns the swid dict, if present """
    swid = {}
    if "swid" in component and "tagId" in component["swid"] and "name" in component["swid"]:
        swid = {val: component["swid"][val] for val in ['tagId', 'name', 'version', 'tagVersion', 'patch', 'text', 'url'] if val in component}
    return swid


def getComponentEvidence(component):
    """ 
    Returns the evidence dict for the component, if any 
    The evidence element was not used to its fullest extent; additional processing is required.
    """
    evidence = {}
    if "evidence" in component:
        evidence_elem = component["evidence"]
        if "identity" in evidence_elem:
            evidence["identity"] = []
            identityList = []
            if isinstance(evidence_elem["identity"], list):
                identityList = evidence_elem["identity"]
            else:
                identityList.append(evidence_elem["identity"])
            for identity in identityList:
                if "field" in identity:
                    identity_dict = {val: identity[val] for val in ['field', 'concludedValue', 'confidence'] if val in identity}
                    if "methods" in identity:
                        identity_dict["methods"] = []
                        for method in identity["methods"]:
                            identity_dict["methods"].append({val: method[val] for val in ['technique', 'confidence', 'value'] if val in method})
                    if "tools" in identity:
                        identity_dict["tools"] = identity["tools"]
                    evidence["identity"].append(identity_dict)
        if "licenses" in evidence_elem:
            evidence["licenses"] = getLicenses(evidence_elem)
        if "copyright" in evidence_elem:
            evidence["copyright"] = []
            for copyright_elem in evidence_elem["copyright"]:
                if "text" in copyright_elem:
                    evidence["copyright"].append(copyright_elem["text"])

    return evidence

## CycloneDX SBOM Ingest

In [ ]:
def analyze_cyclonedx_sbom(sbomFile, plotting = False):
    with open(sbomFile.filepath, 'r') as file:
        sbom_data = json.load(file)
        print(sbomFile.filepath)

    # get components and dependencies
    components = sbom_data.get("components", [])
    dependencies = sbom_data.get("dependencies", [])
    
    # get SBOM header-like information
    sbom_format = sbom_data.get("bomFormat", "")
    sbom_format_version = sbom_data.get("specVersion", "")
    sbom_metadata = sbom_data.get("metadata", {})
    sbom_authors = getAuthors(sbom_metadata)
    sbom_phases = getPhases(sbom_metadata)
    sbom_timestamp = sbom_metadata.get("timestamp", "")
    sbom_serialnumber = sbom_data.get("serialNumber", "")
    sbom_version = sbom_data.get("version", "")
    sbom_supplier = True if "supplier" in sbom_metadata and "name" in sbom_metadata["supplier"] else False
    sbom_target = sbom_metadata.get("component", {})
    target_bomref = sbom_target.get("bom-ref", "")
    target_type = sbom_target.get("type", "")
    target_name = sbom_target.get("name", "")
    target_hash = sbom_target.get("hashes", "")
    target_version = sbom_target.get("version", "")
    target_licenses = getLicenses(sbom_target)
    target_supplier = sbom_target["supplier"]["name"] if "supplier" in sbom_target and "name" in sbom_target["supplier"] else ""
    num_comp = len(components)

    dependency_dict = defaultdict(list)
    graph = nx.DiGraph()
    # iterate through dependencies to build graph of dependency relationships

    count = 0
    node_mapping = {'target': -1}
    for d in dependencies:
        if "ref" in d and "dependsOn" in d:
            source = d['ref']
            if source not in node_mapping.keys():
                node_mapping[source] = f'{count}'
                graph.add_node(node_mapping[source])
                count = count + 1
            for target in d['dependsOn']:
                if target not in node_mapping.keys():
                    node_mapping[target] = f'{count}'
                    graph.add_node(node_mapping[target])
                    graph.add_edge(node_mapping[source], node_mapping[target])
                    count = count + 1
                else:
                    edge = node_mapping[source], node_mapping[target]
                    if not graph.has_edge(*edge):
                        graph.add_edge(*edge)
                    else:
                        continue
    
    # Artifacts of explring the graph structure
    # root_nodes_and_degrees = [[node, graph.out_degree(node)] for node  in graph.nodes if graph.in_degree(node) == 0]
    # root_nodes = [pair[0] for pair in root_nodes_and_degrees]
    # leaf_nodes = [node for node in graph.nodes if graph.out_degree(node) == 0]

    # For building out some graphs, depending on the sbom, you may need to add in the target.
    # graph.add_node(str('target'))
    # for root_node in root_nodes:
    #     graph.add_edge('target', root_node)
    
    # This block of code is for plotting graphs, if plotting = True. Default is False.
    if plotting:
        pos = graphviz_layout(graph, prog = "dot")
        plt.figure(figsize = (40,40))
        nx.draw(graph, pos, with_labels = True, font_size = 35)
        plt.show()

    # are there singletons within the dependencies?
    all_deps = set(dependency_dict.keys()).union(*dependency_dict.values())
    singletons = [comp['bom-ref'] for comp in components if 'bom-ref' in comp and comp['bom-ref'] not in all_deps]

    # is the target part of the dependency tree?
    target_in_deps = target_bomref.strip() != "" and target_bomref in dependency_dict
   
    # calculate depth using longest path algorithms
    try:
        overall_depth = len(nx.dag_longest_path(graph))
    except Exception as e:
        overall_depth = 'None'
        print(f'Exception: {e}. Setting overall depth to "None" and continuing.')

    # breadth: max number of direct dependencies per component
    try:
        max_breadth = max(len(deps) for deps in dependency_dict.values())
    except Exception as e:
        print(f'Exception: {e}. Setting breadth to "None" and continuing.')
        max_breadth = 'None'

    # build dataframe
    component_analysis = []
    for component in components:
        bom_ref = component.get('bom-ref')
        analysis = {
            "component_name": component.get('name'),
            'version': component.get('version'),
            'type': component.get('type'),
            'number_of_direct_dependencies': len(dependency_dict.get(bom_ref, [])),
            'is_singleton': bom_ref in singletons,
            'licenses': ",".join(getLicenses(component)),
            'copyright': component.get("copyright"),
            'number_of hashes': len(getHashes(component)),
            'cpe': component.get('cpe', ''),
            'purl': component.get('purl', ''),
            'swid': json.dumps(getSwid(component)),
            'omniborId': ",".join(component.get('omniborId', [])),
            'swhid': ",".join(component.get('swhid', [])),
            'evidence': json.dumps(getComponentEvidence(component)),
            'submitter': sbomFile.submitter,
            'is_baseline': sbomFile.isBaseline
        }
        component_analysis.append(analysis)


    df = pd.DataFrame(component_analysis)

    # summary data
    summary = {
        "File": os.path.basename(sbomFile.filepath),
        "Total Components": num_comp,
        "Max Depth": overall_depth,
        "Max Breadth": max_breadth,
        "Singletons": len(singletons),
        "Target Type": target_type,
        "Target Name": target_name,
        "Target Hash": target_hash,
        "Target Version": target_version,
        "Target Supplier": target_supplier,
        "Target in Dep Tree": target_in_deps,
        "SBOM Format": sbom_format,
        "SBOM Format Version": sbom_format_version,
        "SBOM Serial Number": sbom_serialnumber,
        "SBOM Version": sbom_version,
        "SBOM Supplier": sbom_supplier,
        "SBOM Timestamp": sbom_timestamp,
        "Target Licenses": ",".join(target_licenses),
        # "SBOM Authors": json.dumps(sbom_authors),
        "SBOM Authors": sbom_authors,
        "SBOM Phases": ",".join(sbom_phases),
        "Submitter": sbomFile.submitter,
        "IsBaseline": sbomFile.isBaseline
    }


    # dataframe of only the distinct component names and versions
    try:
        if not df.empty:
            comp_versions_df = df[['component_name','version']].drop_duplicates()
        else:
            comp_versions_df = pd.DataFrame(columns=['component_name','version'])

    except Exception as e:
        print(f'Exception: {e}. Failed to find comp_versions_df - probably not a valid file {sbomFile.filepath}.')

    return SbomAnalysis(component_df = df, component_versions_df = comp_versions_df, summary = summary, submitter = sbomFile.submitter, isBaseline= sbomFile.isBaseline)

In [ ]:
# Compiles information for component presence across SBOMs

def analyze_cyclonedx_list(list, plotting = False):
    all_dfs = []
    all_comp_versions_df = []
    summary_list = []
    file_count = 0

    # iterate over all json files
    for sbomFile in list:
        try:
            sbom = analyze_cyclonedx_sbom(sbomFile, plotting)
        except Exception as e:
            print(f"Exception in analyze_cyclonedx_sbom: {e}")
            raise e
        else:
            summary_list.append(sbom.summary)
            sbom.component_df["SBOM File"] = os.path.basename(sbomFile.filepath)
            all_dfs.append(sbom.component_df)
            all_comp_versions_df.append(sbom.component_versions_df)
            file_count = file_count + 1
    try:
        combined_df = pd.concat(all_dfs, ignore_index = True)
        combined_comp_versions_df = pd.concat(all_comp_versions_df, ignore_index = True)
        summary_df = pd.DataFrame(summary_list)
    except Exception as e:
        combined_df = pd.DataFrame()
        combined_comp_versions_df = pd.DataFrame()
        summary_df = pd.DataFrame()
        print(f"Exception: {e}")

    return TargetAnalysis(target_component_df = combined_df, target_component_versions_df = combined_comp_versions_df, 
                          target_summary = summary_df, target_file_count = file_count)

In [ ]:
def get_component_overlap(comp_combined_df, file_count):
    """Returns a df with the percentage of each component by name and version shared among files"""
    df_subset = comp_combined_df[['component_name', 'version']].dropna()
    overlapSeries = df_subset.value_counts() * 100 / file_count
    overlapSeries.apply(lambda x: round(x, 1))
    return overlapSeries

In [ ]:
# Establish path structure and extract source and build data 
# we extracted and saved the path and filenames to facilitate data quality checks
targets = ['dependency_track', 'gin','hexyl', 'httpie_cli', 'jq', 'minecolonies', 'nodejs-goof', 'opencv', 'phpmailer']
source_path_template = '../submissions_by_target/?/source/cyclone'
build_path_template = '../submissions_by_target/?/build/cyclone'

source_paths, build_paths = {}, {}

for target in targets:
    source_paths[target] = []
    build_paths[target] = []

for target in targets:
    source_path = source_path_template.replace('?', target)
    build_path = build_path_template.replace('?', target)
    if os.path.isdir(source_path):
        for filename in os.listdir(source_path):
            if filename.endswith(".json"):
                source_paths[target].append(SbomFile(
                    format = SbomFormat.JSON,
                    phase = SbomPhase.SOURCE,
                    standard = SbomStandard.CYCLONEDX,
                    target = SbomTarget.from_name(target),
                    filepath = os.path.join(source_path, filename),
                    submitter = "",
                    isBaseline = False
                ))
    if os.path.isdir(build_path):
        for filename in os.listdir(build_path):
            if filename.endswith(".json"):
                build_paths[target].append(SbomFile(
                    format = SbomFormat.JSON,
                    phase = SbomPhase.BUILD,
                    standard = SbomStandard.CYCLONEDX,
                    target = SbomTarget.from_name(target),
                    filepath = os.path.join(build_path, filename),
                    submitter = "",
                    isBaseline = False
                ))

source_sboms = {target: analyze_cyclonedx_list(source_paths[target], plotting = False) for target in source_paths.keys() if source_paths[target]}
build_sboms = {target: analyze_cyclonedx_list(build_paths[target], plotting = False) for target in build_paths.keys() if build_paths[target]}

## Submitted SBOM Analysis

In [ ]:
# start building out the output directories

cmp_similarity_dir = cyclone_output_dir + '/' + component_similarity_dir + '/'
output_summary_stats_dir = cyclone_output_dir + '/' + summary_stats_dir + '/'
ind_compinfo_dir = cyclone_output_dir + '/' + individual_info_dir + '/'

Path(cmp_similarity_dir).mkdir(parents=True, exist_ok=True)
Path(output_summary_stats_dir).mkdir(parents=True, exist_ok=True)
Path(ind_compinfo_dir).mkdir(parents=True, exist_ok=True)

In [ ]:
# see preamble for output file definitions

source_target_component_overlaps = {}
build_target_component_overlaps = {}
source_combined_csv_template = 'src_full_combined_?.csv' 
build_combined_csv_template =  cmp_similarity_dir +'bld_full_combined_?.csv' 
source_overlap_csv_template = cmp_similarity_dir + 'src_comp_version_?.csv' 
build_overlap_csv_template = cmp_similarity_dir + 'bld_comp_version_?.csv'

# for each source sbom, determine component overlap
for target, result in source_sboms.items():
    try:
        source_target_component_overlaps[target] = get_component_overlap(result.target_component_versions_df, result.target_file_count)
        df = source_target_component_overlaps[target].to_frame()
        df.rename(columns={df.columns[0]: "Percent of SBOMs" }, inplace = True)
        df['Number of Files'] = result.target_file_count
        # result.target_component_df.to_csv(source_combined_csv_template.replace('?', target))
        df.to_csv(source_overlap_csv_template.replace('?', target))
    except Exception as e:
        print(f"Exception: {e}")

# for each build sbom, determine component overlap
for target, result in build_sboms.items():
    try:
        build_target_component_overlaps[target] = get_component_overlap(result.target_component_versions_df, result.target_file_count)
        df = build_target_component_overlaps[target].to_frame()
        df.rename(columns={df.columns[0]: "Percent of SBOMs" }, inplace = True)
        df['Number of Files'] = result.target_file_count
        # result.target_component_df.to_csv(build_combined_csv_template.replace('?', target))
        df.to_csv(build_overlap_csv_template.replace('?', target))
    except Exception as e:
        print(f"Exception: {e}")

In [ ]:
# High-level summary statistics/data for each SBOM (mostly minimum elements)
merged_summary_df = pd.DataFrame()
merged_filename = output_summary_stats_dir + 'merged_cyclonedx_summary_files.csv'

for target, result in source_sboms.items():
    filename = output_summary_stats_dir + 'source' + f'_{target}_cyclonedx_' + 'summary.csv'
    result.target_summary.to_csv(filename)
    merged_summary_df = pd.concat([merged_summary_df, result.target_summary])

for target, result in build_sboms.items():
    filename = output_summary_stats_dir + 'build' + f'_{target}_cyclonedx_' + 'summary.csv'
    result.target_summary.to_csv(filename)
    merged_summary_df = pd.concat([merged_summary_df, result.target_summary])

merged_summary_df.to_csv(merged_filename)


In [ ]:
# component-level information

merged_component_df = pd.DataFrame()
merged_component_filename = output_summary_stats_dir + 'merged_cyclonedx_component_data.csv'

for target, result in source_sboms.items():
    filename = ind_compinfo_dir + 'source' + f'_{target}_cyclonedx_' + 'component_info.csv'
    result.target_component_df.to_csv(filename)
    merged_component_df = pd.concat([merged_component_df, result.target_component_df])


for target, result in build_sboms.items():
    filename = ind_compinfo_dir + 'build' + f'_{target}_cyclonedx_' + 'component_info.csv'
    result.target_component_df.to_csv(filename)
    merged_component_df = pd.concat([merged_component_df, result.target_component_df])

merged_summary_df.to_csv(merged_component_filename)


In [ ]:
# merge files
# Initialize storage dataframe to be empty

component_files_df = pd.DataFrame()

# read all CSVs in component_similarity
for filename in os.listdir(cmp_similarity_dir):
    if filename.endswith(".csv"):
        filepath = os.path.join(cmp_similarity_dir, filename)

        df = pd.read_csv(filepath)

        # filename structure: [build/src]_comp_version_[target].csv
        name_parts = filename.split('_')
        target_part = name_parts[3].split('.')
        target = target_part[0] if len(target_part) > 0 else ''
        build_source = name_parts[0] if len(name_parts) > 1 else ''

        df['Target'] = target
        df['Build or Source'] = build_source

        component_files_df = pd.concat([component_files_df, df], ignore_index=True)

# write merged CycloneDX component file
component_files_df.to_csv(cmp_similarity_dir + 'merged_cycloneDX_component_similarity_files.csv')


In [ ]:
# spot checking of outputs
for target, result in source_target_component_overlaps.items():
    print('Source: ' + target + '\n')
    print(result)
    #filename = 'source' + f'_{target}_cyclonedx_' + 'component_overlaps.csv'
    #result.target_component_df.to_csv(filename)
for target, result in build_target_component_overlaps.items():
    print('Build: ' + target + '\n')
    print(result) 
    #filename = 'build' + f'_{target}_cyclonedx_' + 'component_overlaps.csv'   

In [ ]:
# spot checking of outputs
for target, result in source_target_component_overlaps.items():    
    print('Source: ' + target + '\n')
    print(result)
    #filename = 'source' + f'_{target}_cyclonedx_' + 'component_overlaps.csv'
    #result.target_component_df.to_csv(filename)
for target, result in build_target_component_overlaps.items():
    print('Build: ' + target + '\n')
    print(result) 
    #filename = 'build' + f'_{target}_cyclonedx_' + 'component_overlaps.csv'   

## Baseline SBOM Analysis

In [ ]:
baseline_source_paths, baseline_build_paths = {}, {}

for target in all_targets:
    baseline_source_paths[target] = []
    baseline_build_paths[target] = []

# walk the sbom directory and test each file name against `file_pattern`
for sbom_dir in [sbom_baseline_dir]:
    for root, dirs, files in os.walk(sbom_dir):
        for file in files:
            sbomFile = sbomInfoFromFile(os.path.join(root, file))
            if sbomFile.isValid() and sbomFile.standard.isCycloneDx() and sbomFile.format.isJson():
                if sbomFile.phase is SbomPhase.SOURCE:
                    baseline_source_paths[sbomFile.target.value].append(sbomFile)
                    # print(sbomFile)
                elif sbomFile.phase is SbomPhase.BUILD:
                    baseline_build_paths[sbomFile.target.value].append(sbomFile)
                    # print(sbomFile)
baseline_source_sboms = {target: analyze_cyclonedx_list(baseline_source_paths[target]) for target in baseline_source_paths.keys() if baseline_source_paths[target]}
baseline_build_sboms = {target: analyze_cyclonedx_list(baseline_build_paths[target]) for target in baseline_build_paths.keys() if baseline_build_paths[target]}

In [ ]:
baseline_source_target_component_overlaps = {}
baseline_build_target_component_overlaps = {}
baseline_cmp_similarity_dir = '../outputs/' + cyclone_baseline_output_dir + '/' + component_similarity_dir + '/'
cmp_similarity_dir = cyclone_output_dir + '/' + component_similarity_dir + '/'

Path(baseline_cmp_similarity_dir).mkdir(parents=True, exist_ok=True)
Path(cmp_similarity_dir).mkdir(parents=True, exist_ok=True)

build_combined_csv_template =  baseline_cmp_similarity_dir +'bld_full_combined_?.csv'
source_overlap_csv_template = baseline_cmp_similarity_dir + 'src_comp_version_?.csv'
build_overlap_csv_template = baseline_cmp_similarity_dir + 'bld_comp_version_?.csv'


# for each source sbom, determine component overlap
for target, result in baseline_source_sboms.items():
    try:
        baseline_source_target_component_overlaps[target] = get_component_overlap(result.target_component_versions_df, result.target_file_count)
        df = baseline_source_target_component_overlaps[target].to_frame()
        df.rename(columns={df.columns[0]: "Percent of SBOMs" }, inplace = True)
        df['Number of Files'] = result.target_file_count
        df.to_csv(source_overlap_csv_template.replace('?', target))
    except Exception as e:
        print(f"Exception: {e}")

# for each build sbom, determine component overlap
for target, result in baseline_build_sboms.items():
    try:
        baseline_build_target_component_overlaps[target] = get_component_overlap(result.target_component_versions_df, result.target_file_count)
        df = build_target_component_overlaps[target].to_frame()
        df.rename(columns={df.columns[0]: "Percent of SBOMs" }, inplace = True)
        df['Number of Files'] = result.target_file_count
        df.to_csv(build_overlap_csv_template.replace('?', target))
    except Exception as e:
        print(f"Exception: {e}")

In [ ]:
output_summary_stats_dir = '../outputs/' + cyclone_baseline_output_dir + '/' + summary_stats_dir + '/'

Path(output_summary_stats_dir).mkdir(parents=True, exist_ok=True)

merged_baseline_summary_df = pd.DataFrame()
merged_baseline_filename = output_summary_stats_dir + 'merged_baseline_cyclonedx_summary_files.csv'


for target, result in baseline_source_sboms.items():
    filename = output_summary_stats_dir + 'source' + f'_{target}_cyclonedx_' + 'summary.csv'
    result.target_summary.to_csv(filename)
    merged_baseline_summary_df = pd.concat([merged_baseline_summary_df, result.target_summary])

for target, result in baseline_build_sboms.items():
    filename = output_summary_stats_dir + 'build' + f'_{target}_cyclonedx_' + 'summary.csv'
    result.target_summary.to_csv(filename)
    merged_baseline_summary_df = pd.concat([merged_baseline_summary_df, result.target_summary])

merged_baseline_summary_df.to_csv(merged_baseline_filename)


In [ ]:
baseline_ind_compinfo_dir = '../outputs/' + cyclone_baseline_output_dir + '/' + individual_info_dir + '/'

Path(baseline_ind_compinfo_dir).mkdir(parents=True, exist_ok=True)

merged_baseline_component_df = pd.DataFrame()
merged_baseline_component_filename = baseline_ind_compinfo_dir + 'merged_baseline_cyclonedx_component_files.csv'

for target, result in baseline_source_sboms.items():
    filename = baseline_ind_compinfo_dir + 'source' + f'_{target}_cyclonedx_' + 'component_info.csv'
    result.target_component_df.to_csv(filename)
    merged_baseline_component_df = pd.concat([merged_baseline_component_df, result.target_component_df])


for target, result in baseline_build_sboms.items():
    filename = baseline_ind_compinfo_dir + 'build' + f'_{target}_cyclonedx_' + 'component_info.csv'
    result.target_component_df.to_csv(filename)
    merged_baseline_component_df = pd.concat([merged_baseline_component_df, result.target_component_df])

merged_baseline_component_df.to_csv(merged_baseline_component_filename)
print(individual_info_dir)

